
# Accelerated probabilistic tracking

DIPY's accelerated tracker :footcite:p:`Kruper2025` is a stripped-down
probabilistic tracker that runs on various hardware through one of three backends:
CUDA (NVIDIA), Metal (Apple Silicon) or WebGPU (any accelerator with subgroup
support). It generates tens of millions of streamlines in minutes, at the
price of a few restrictions compared with
:func:`dipy.tracking.tracker.probabilistic_tracking` on the CPU:

- the whole PMF (spherical function) must fit in memory, so the ODF is
  sampled on a sphere rather than kept as spherical harmonics;
- voxels must be isotropic;
- the stopping criterion must be a
  :class:`dipy.tracking.stopping_criterion.ThresholdStoppingCriterion`;
- the streamline length is capped at a fixed number of steps.

The accelerated dependencies are optional. Install the one matching your hardware::

    pip install "dipy[cu12]"     # or dipy[cu13], for CUDA 12 / 13
    pip install "dipy[metal]"    # Apple Silicon
    pip install "dipy[webgpu]"   # cross-platform, needs a wgpu adapter

This example follows `sphx_glr_examples_built_fiber_tracking_tracking_probabilistic.py`
and only changes the tracking call and number of seeds. We start by fitting a CSD model.


In [ ]:
import numpy as np

from dipy.core.gradients import gradient_table
from dipy.data import default_sphere, get_fnames
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti, load_nifti_data
from dipy.io.stateful_tractogram import Space, StatefulTractogram
from dipy.io.streamline import save_tractogram, save_trx_from_generator
from dipy.reconst.csdeconv import ConstrainedSphericalDeconvModel, auto_response_ssst
from dipy.reconst.dti import TensorModel
from dipy.tracking.accel import select_accel_backend
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking.streamline import Streamlines
from dipy.tracking.tracker import probabilistic_tracking
from dipy.tracking.utils import random_seeds_from_mask
from dipy.viz import has_fury

if has_fury:
    from fury import actor, colormap, window

# Enables/disables interactive visualization
interactive = False

hardi_fname, hardi_bval_fname, hardi_bvec_fname = get_fnames(name="stanford_hardi")
label_fname = get_fnames(name="stanford_labels")

data, affine, hardi_img = load_nifti(hardi_fname, return_img=True)
labels = load_nifti_data(label_fname)
bvals, bvecs = read_bvals_bvecs(hardi_bval_fname, hardi_bvec_fname)
gtab = gradient_table(bvals, bvecs=bvecs)

white_matter = (labels == 1) | (labels == 2)
seed_mask = labels == 2
seeds = random_seeds_from_mask(
    seed_mask, affine, seeds_count=20, seed_count_per_voxel=True, random_seed=1
)

response, ratio = auto_response_ssst(gtab, data, roi_radii=10, fa_thr=0.7)
csd_model = ConstrainedSphericalDeconvModel(gtab, response, sh_order_max=6)
csd_fit = csd_model.fit(data, mask=white_matter)

The accelerated tracker stops on a scalar map. Here we use the FA from a
tensor fit, thresholded at 0.2.



In [ ]:
tensor_model = TensorModel(gtab)
tensor_fit = tensor_model.fit(data, mask=white_matter)
stopping_criterion = ThresholdStoppingCriterion(tensor_fit.fa, 0.2)

``backend="auto"`` (the default) picks the first available GPU backend, in
this order: Metal, CUDA, then WebGPU, and falls back to the CPU when there
is none or when the inputs need the CPU tracker (e.g. another stopping
criterion). ``select_accel_backend`` tells which GPU backend would be
used; pass ``backend="metal"``, ``"cuda"``, ``"webgpu"`` or ``"cpu"`` to
force one.



In [ ]:
backend = select_accel_backend() or "cpu"
print(f"Tracking with the {backend} backend")

The tracking call is the same as on the CPU.
Passing ``sh`` is fine: the coefficients are sampled on ``sphere`` before
being uploaded to the device.



In [ ]:
streamline_generator = probabilistic_tracking(
    seeds,
    stopping_criterion,
    affine,
    sh=csd_fit.shm_coeff,
    sphere=default_sphere,
    random_seed=1,
    max_angle=20,
    step_size=0.5,
    backend="auto",
)

streamlines = Streamlines(streamline_generator)
sft = StatefulTractogram(streamlines, hardi_img, Space.RASMM)
save_tractogram(sft, "tractogram_accel.trx")

if has_fury:
    colors = np.repeat(
        colormap.line_colors(streamlines), [len(sl) for sl in streamlines], axis=0
    )
    lines_actor = actor.streamlines(lines=streamlines, colors=colors)
    window.snapshot(actors=[lines_actor], fname="tractogram_accel.png")
    if interactive:
        window.show([lines_actor])

.. rst-class:: centered small fst-italic fw-semibold

Using TRX with the accelerated probabilistic tracker



The accelerated tracker produces streamlines much faster than they
can be collected one at a time in Python. With ``chunked=True`` the
generator yields one ``(points, lengths)`` tuple per chunk of ``chunk_size``
seeds instead, and :func:`dipy.io.streamline.save_trx_from_generator`
writes those chunks straight to a TRX file, so whole-brain tractograms
of hundreds of gigabytes never have to fit in memory.



In [ ]:
streamline_generator = probabilistic_tracking(
    seeds,
    stopping_criterion,
    affine,
    sh=csd_fit.shm_coeff,
    sphere=default_sphere,
    random_seed=1,
    max_angle=20,
    step_size=0.5,
    backend="auto",
    chunked=True,
    chunk_size=25000,
)

trx = save_trx_from_generator(
    streamline_generator,
    hardi_img,
    filename="tractogram_accel_chunked.trx",
    chunked=True,
    nb_streamlines_estimate=len(seeds),
)
print(f"Saved {trx.header['NB_STREAMLINES']} streamlines")
trx.close()

The ``dipy_track`` command line tool uses the same chunked writer whenever
the output tractogram is a ``.trx`` file.



## References

.. footbibliography::




.. include:: ../../links_names.inc


